## 과제 2: 카카오 맵 API를 활용한 할리스 카페 위치 정보 수집

### 개요

카카오 맵 API를 활용하여 할리스 카페 매장의 주소를 위도와 경도 좌표로 변환(지오코딩)하고, 이 데이터를 수집하여 저장하는 과제입니다.

### 준비사항

1. 카카오 개발자 사이트(https://developers.kakao.com) 회원가입 및 로그인
2. 애플리케이션 등록 및 API 키 발급
3. REST API 키 확인
4. Python 설치 및 requests 라이브러리 설치
5. VSCode 주피터 환경 설치

### 과제 요구사항

1. 카카오 로컬 API의 주소 검색 기능 API활용 방법에 대해 숙지하세요.
    - https://developers.kakao.com/docs/latest/ko/local/dev-guide
2. 카카오 로컬 API의 주소 검색 기능을 활용하여:
    - 제공된 할리스 카페 주소 목록에 대한 위도, 경도 정보 수집  --> **주소로 좌표변환**
        
        [hollys_stores.csv](attachment:e9b626ae-c4cc-4406-9e97-c9b66e88169c:hollys_stores.csv)
        
    - 주소, 매장명, 위도, 경도를 포함한 데이터 저장
    - 오류 발생 시 적절한 예외 처리 구현
3. 수집한 데이터 처리:
    - CSV 파일로 정리하여 저장
    - 선택사항: 수집한 좌표를 지도에 표시(folium 라이브러리 활용)

In [1]:
# API 접근을 위한 클라이언트 정보 설정
API_KEY = "213a9bc50d06f380a3bc47a6f09e517d"

In [2]:
# pip install folium

In [5]:
# 카카오 맵 API를 활용한 할리스 카페 위치 정보 수집
import requests
import pandas as pd
import time
import os
import re
from datetime import datetime
import folium
from folium.plugins import MarkerCluster


# # API 키 설정
# API_KEY = "YOUR_API_KEY"  # 카카오 개발자 사이트에서 발급받은 REST API 키로 변경

# 헤더 설정
headers = {
    "Authorization": f"KakaoAK {API_KEY}"
}

# 지오코딩 API 엔드포인트 (주소로 좌표 변환)
api_url = "https://dapi.kakao.com/v2/local/search/address.json"


def geocode_address(address):
    """
    주소를 위도와 경도로 변환하는 함수
    
    Args:
        address (str): 지오코딩할 주소
        
    Returns:
        dict or None: 위도, 경도 정보를 담은 딕셔너리, 오류 시 None 반환
    """
    cleaned = clean_address(address)
    params = {"query": cleaned}
    
    try:
        response = requests.get(api_url, headers=headers, params=params)
        
        # API 응답 상태 확인
        if response.status_code == 200:
            result = response.json()
            
            # 검색 결과가 있는 경우
            if result.get("documents"):
                # 첫 번째 결과 사용
                location = result["documents"][0]
                
                # 도로명 주소가 있으면 도로명 주소 사용, 없으면 지번 주소 사용
                road_address = location.get("road_address")
                if road_address:
                    address_name = road_address.get("address_name", "")
                else:
                    address_name = location.get("address", {}).get("address_name", "")
                
                # 위도, 경도 반환
                return {
                    "lat": float(location["y"]),  # 위도
                    "lng": float(location["x"]),  # 경도
                    "address_found": address_name  # 찾은 주소
                }
            else:
                print(f"주소를 찾을 수 없음: {cleaned}")
                return None
        else:
            print(f"API 요청 실패: {response.status_code} - {response.text}")
            
            # 429 에러(Too Many Requests)인 경우 재시도 안내
            if response.status_code == 429:
                print("API 호출 한도를 초과했습니다. 잠시 후 다시 시도하세요.")
            
            return None
            
    except Exception as e:
        print(f"지오코딩 중 오류 발생: {e}")
        return None
    
    
def clean_address(addr):
    if not isinstance(addr, str):
        return ""
    # 층 제거
    addr = re.sub(r"\d+ ?층|지하\d+층|B\d+|[0-9]+~[0-9]+층", "", addr)
    return addr

In [ ]:

hollys = pd.read_csv("../../data/hollys_stores.csv", encoding="utf-8-sig")

lats, lngs = [], []
for i in range(len(hollys)):
    addr = hollys.loc[i, "address"]
    result = geocode_address(addr)
    
    if result :
        lats.append(result['lat'])
        lngs.append(result['lng'])
    else:
        lats.append(None)
        lngs.append(None)
    
hollys['lat']= lats
hollys['lng']= lngs

hollys.head()
# hollys.to_csv("hollys_stores_latlng.csv")

,region,name,status,address,service,phone,lat,lng
0,서울 동작구,신대방삼거리역점,영업중,서울특별시 동작구 상도로 60 (대방동) 1층~2층,NaN,02-823-2377,37.499561,126.927153
1,전북 익산시,익산부송점,영업중,"전라북도 익산시 하나로10길 70-1 (부송동, 레스트빌딩 1층) .",주차,063-832-1717,35.961406,126.989041
2,경남 창원시 진해구,진해속천점,영업중,경상남도 창원시 진해구 태평로 132 (속천동) 2~3층,테라스 주차,070-7786-1019,35.141675,128.670750
3,서울 영등포구,여의도포스트타워점,영업중,서울특별시 영등포구 여의나루로 60 포스트타워 여의도 1층,주차,02-2135-5321,37.522300,126.926225
4,서울 영등포구,대림역점,영업중,서울특별시 영등포구 도림로 140 대림빌딩 101호,테라스,02-834-1000,37.493024,126.897628


In [7]:
from IPython.display import display

valid = hollys.dropna(subset=["lat", "lng"])
m = folium.Map(location=[valid["lat"].mean(), valid["lng"].mean()], zoom_start=11)

# iterrows(): DataFrame의 각 행(row)을 순서대로 하나씩 꺼냄
# index는 행의 인덱스(index)
# row은 그 행의 데이터(Series 형태)
for index, row in valid.iterrows():
    folium.Marker(
        location=[row['lat'], row['lng']],
        color='blue',
        fill=True,
        fill_color='blue',
    ).add_to(m)

# 지도 저장
# seoul_map.save('stores_map.html')

m


display(m)

In [ ]:
hollys

,region,name,status,address,service,phone,lat,lng
0,서울 동작구,신대방삼거리역점,영업중,서울특별시 동작구 상도로 60 (대방동) 1층~2층,NaN,02-823-2377,37.499561,126.927153
1,전북 익산시,익산부송점,영업중,"전라북도 익산시 하나로10길 70-1 (부송동, 레스트빌딩 1층) .",주차,063-832-1717,35.961406,126.989041
2,경남 창원시 진해구,진해속천점,영업중,경상남도 창원시 진해구 태평로 132 (속천동) 2~3층,테라스 주차,070-7786-1019,NaN,NaN
3,서울 영등포구,여의도포스트타워점,영업중,서울특별시 영등포구 여의나루로 60 포스트타워 여의도 1층,주차,02-2135-5321,37.522300,126.926225
4,서울 영등포구,대림역점,영업중,서울특별시 영등포구 도림로 140 대림빌딩 101호,테라스,02-834-1000,37.493024,126.897628
5,부산 동래구,부산동래온천점,영업중,부산광역시 동래구 금강로 116 (온천동) 금강메디컬타워 1층,주차,051-923-5538,35.218985,129.079913
6,서울 서대문구,충정로역점,영업중,서울특별시 서대문구 경기대로 26-26 어바니엘충정로 201동 지하1층 102호,주차,02-383-0807,37.561023,126.962192
7,부산 사하구,부산고신대의과대학점,영업중,부산광역시 서구 감천로 262 고신의대 강의동 1층,주차,051-710-7565,35.081078,129.014513
8,대구 수성구,대구수성못DI점,영업중,대구광역시 수성구 용학로 62 1~2층,테라스 주차,053-761-7186,NaN,NaN
9,경남 진주시,경상국립대학생회관점,영업중,경상남도 진주시 진주대로 501 경상국립대 학생회관 1층,NaN,055-772-0931,35.156714,128.098001


---

# 설튜터님.ver

In [ ]:
# 카카오 맵 API를 활용한 할리스 카페 위치 정보 수집
import requests
import pandas as pd
import time
import os
from datetime import datetime
import folium
from folium.plugins import MarkerCluster

# API 키 설정
API_KEY = "YOUR_API_KEY"  # 카카오 개발자 사이트에서 발급받은 REST API 키로 변경

# 헤더 설정
headers = {
    "Authorization": f"KakaoAK {API_KEY}"
}

# 지오코딩 API 엔드포인트
api_url = "https://dapi.kakao.com/v2/local/search/address.json"


def geocode_address(address):
    """
    주소를 위도와 경도로 변환하는 함수
    
    Args:
        address (str): 지오코딩할 주소
        
    Returns:
        dict or None: 위도, 경도 정보를 담은 딕셔너리, 오류 시 None 반환
    """
    params = {"query": address}
    
    try:
        response = requests.get(api_url, headers=headers, params=params)
        
        # API 응답 상태 확인
        if response.status_code == 200:
            result = response.json()
            
            # 검색 결과가 있는 경우
            if result.get("documents"):
                # 첫 번째 결과 사용
                location = result["documents"][0]
                
                # 도로명 주소가 있으면 도로명 주소 사용, 없으면 지번 주소 사용
                road_address = location.get("road_address")
                if road_address:
                    address_name = road_address.get("address_name", "")
                else:
                    address_name = location.get("address", {}).get("address_name", "")
                
                # 위도, 경도 반환
                return {
                    "lat": float(location["y"]),  # 위도
                    "lng": float(location["x"]),  # 경도
                    "address_found": address_name  # 찾은 주소
                }
            else:
                print(f"주소를 찾을 수 없음: {address}")
                return None
        else:
            print(f"API 요청 실패: {response.status_code} - {response.text}")
            
            # 429 에러(Too Many Requests)인 경우 재시도 안내
            if response.status_code == 429:
                print("API 호출 한도를 초과했습니다. 잠시 후 다시 시도하세요.")
            
            return None
            
    except Exception as e:
        print(f"지오코딩 중 오류 발생: {e}")
        return None

def process_hollys_stores(csv_file):
    """
    할리스 매장 CSV 파일을 처리하여 위도, 경도 정보를 추가하는 함수
    
    Args:
        csv_file (str): 할리스 매장 정보가 담긴 CSV 파일 경로
        
    Returns:
        pandas.DataFrame: 위도, 경도 정보가 추가된 데이터프레임
    """
    try:
        # CSV 파일 읽기
        df = pd.read_csv(csv_file)
        print(f"CSV 파일 로드 완료: 총 {len(df)} 개의 매장 정보")
        
        # 위도, 경도 열 추가
        df["latitude"] = None
        df["longitude"] = None
        df["address_found"] = None
        df["geocoding_success"] = False
        
        # 각 매장 주소에 대해 지오코딩 실행
        for idx, row in df.iterrows():
            address = row["address"]
            name = row["name"]
            
            print(f"처리 중 ({idx+1}/{len(df)}): {name} - {address}")
            
            # 지오코딩
            location = geocode_address(address)
            
            # 결과 저장
            if location:
                df.at[idx, "latitude"] = location["lat"]
                df.at[idx, "longitude"] = location["lng"]
                df.at[idx, "address_found"] = location["address_found"]
                df.at[idx, "geocoding_success"] = True
                print(f"  주소:  location['address_found']")
                print(f"  성공: 위도={location['lat']}, 경도={location['lng']}")
            else:
                print(f"  실패: 위치 정보를 찾을 수 없음")
            
            # API 호출 제한을 고려해 잠시 대기 (초당 요청 수 제한 방지)
            time.sleep(0.5)
        
        return df
    
    except Exception as e:
        print(f"CSV 파일 처리 중 오류 발생: {e}")
        return None

def create_map(df, output_file):
    """
    위도, 경도 정보를 이용해 할리스 매장 위치를 지도에 표시하는 함수
    
    Args:
        df (pandas.DataFrame): 위도, 경도 정보가 포함된 데이터프레임
        output_file (str): 저장할 HTML 파일 경로
        
    Returns:
        bool: 지도 생성 성공 여부
    """
    try:
        # 성공적으로 지오코딩된 매장만 필터링
        success_df = df[df["geocoding_success"] == True].copy()
        
        if len(success_df) == 0:
            print("지도에 표시할 위치 정보가 없습니다.")
            return False
        
        # 모든 매장의 중심점 계산
        center_lat = success_df["latitude"].mean()
        center_lng = success_df["longitude"].mean()
        
        # 지도 생성
        m = folium.Map(location=[center_lat, center_lng], zoom_start=7)
        
        # 매장이 많을 경우를 대비해 마커 클러스터 사용
        marker_cluster = MarkerCluster().add_to(m)
        
        # 각 매장을 지도에 추가
        for idx, row in success_df.iterrows():
            # 팝업 내용 생성
            popup_text = f"""
            <b>{row['name']}</b><br>
            주소: {row['address']}<br>
            전화: {row['phone']}<br>
            서비스: {row['service']}
            """
            
            # 마커 추가
            folium.Marker(
                location=[row["latitude"], row["longitude"]],
                popup=folium.Popup(popup_text, max_width=300),
                tooltip=row["name"],
                icon=folium.Icon(color="red", icon="coffee", prefix="fa")
            ).add_to(marker_cluster)
        
        # 지도 저장
        m.save(output_file)
        print(f"지도를 저장했습니다: {output_file}")
        return True
    
    except Exception as e:
        print(f"지도 생성 중 오류 발생: {e}")
        return False

In [ ]:
# 입력 파일
input_file = "hollys_stores.csv"

# 출력 파일 경로 설정
output_dir = "./hollys_locations"
os.makedirs(output_dir, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S") # 현재 시간을 파일명에 포함
output_csv = os.path.join(output_dir, f"hollys_locations_{timestamp}.csv")
output_map = os.path.join(output_dir, f"hollys_map_{timestamp}.html")

print("할리스 카페 위치 정보 수집 시작")
print("-" * 50)

# 1. CSV 파일 처리 및 지오코딩
df = process_hollys_stores(input_file)

# 2. 결과 CSV 저장
df.to_csv(output_csv, index=False, encoding="utf-8-sig")
print(f"위치 정보가 포함된 CSV 파일을 저장했습니다: {output_csv}")

# 3. 지오코딩 결과 통계
total = len(df)
success = df["geocoding_success"].sum()
fail = total - success

print("-" * 50)
print(f"지오코딩 결과 요약:")
print(f"- 총 매장 수: {total}")
print(f"- 성공: {success} ({success/total*100:.1f}%)")
print(f"- 실패: {fail} ({fail/total*100:.1f}%)")

# 4. 지도 생성 (folium 라이브러리 사용)
print("-" * 50)
print("지도 생성 중...")
create_map(df, output_map)

print("-" * 50)
print("처리 완료!")
print(f"- CSV 파일: {output_csv}")
print(f"- 지도 파일: {output_map}")
